In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os  # import modules

In [ ]:
plt.rcParams['font.family'] = 'AppleGothic'

In [ ]:
jeju_reg_17_df = pd.read_csv(os.path.join("data","jeju_card_region_2017.csv"))
jeju_reg_18_df = pd.read_csv(os.path.join("data","jeju_card_region_2018.csv"))
jeju_pop_df = pd.read_csv(os.path.join("data","jeju_population.csv"))

In [ ]:
jeju_reg_17_df.head()

In [ ]:
jeju_reg_18_df.head()

In [ ]:
jeju_pop_df.head()

In [ ]:
jeju_reg_17_df.info()

In [ ]:
jeju_reg_18_df.info()

In [ ]:
jeju_pop_df.info()

In [ ]:
jeju_reg_17_df.describe(include='all')

In [ ]:
pd.options.display.float_format = '{:.3f}'.format # Format numbers in fixed-point notation for readability

In [ ]:
jeju_reg_17_df.describe(include='all')

In [ ]:
jeju_reg_18_df.describe(include='all')

In [ ]:
jeju_pop_df.describe(include='all')

In [ ]:
def print_unique_values(df):  # Print unique values for all string columns
    object_columns = df.columns[df.dtypes == 'str']
    for col in object_columns:
        print(f'{col} check number of unique values per column: {df[col].nunique()}')
        print(sorted(df[col].unique()), '\n')

In [ ]:
print_unique_values(jeju_reg_17_df)

In [ ]:
print_unique_values(jeju_reg_18_df)

In [ ]:
for item in jeju_reg_17_df['업종명'].unique():    # Check for industry categories that exist in only one year
    if item not in jeju_reg_18_df['업종명'].unique():
        print(f'Only in 2017: {item}')
        
for item in jeju_reg_18_df['업종명'].unique():
    if item not in jeju_reg_17_df['업종명'].unique():
        print(f'Only in 2018: {item}')

In [ ]:
print(jeju_reg_17_df[jeju_reg_17_df["업종명"] == "기타 갬블링 및 베팅업"].shape)

In [ ]:
print(jeju_reg_18_df[jeju_reg_18_df["업종명"] == "택시 운송업"].shape)

In [ ]:
jeju_reg_17_df = jeju_reg_17_df[jeju_reg_17_df['업종명'] != '기타 갬블링 및 베팅업']
jeju_reg_18_df = jeju_reg_18_df[jeju_reg_18_df['업종명'] != '택시 운송업'] # Remove categories that don't appear in both years

In [ ]:
for item in jeju_reg_17_df['읍면동명'].unique(): # Check for area categories that exist in only one yea
    if item not in jeju_reg_18_df['읍면동명'].unique():
        print(f'Only in 2017: {item}')
        
for item in jeju_reg_18_df['읍면동명'].unique():
    if item not in jeju_reg_17_df['읍면동명'].unique():
        print(f'Only in 2018: {item}')

In [ ]:
# Merge datasets
jeju_reg_df = pd.concat([jeju_reg_17_df, jeju_reg_18_df])

In [ ]:
jeju_reg_df.shape

In [ ]:
jeju_reg_df['연월'].unique()

In [ ]:
jeju_reg_df['연월'] = jeju_reg_df['연월'].str[:7] # Get rid of day since it's all 01
jeju_reg_df.head()

In [ ]:
# -----------------------------------Data exploration and preprocessing complete----------------------------

In [ ]:
groupby_sector = jeju_reg_df.groupby("업종명").sum(numeric_only = True) # Group by industry and calculate total sum

In [ ]:
groupby_sector.sort_values(by="이용금액", ascending = False).head(10) # Get top 10 industries by total sales amount

In [ ]:
# Korean restaurants rank 1st in total sales

In [ ]:
groupby_sector.sort_values(by="이용자수", ascending = False).head(10) # Get top 10 industries by total customers

In [ ]:
# Convenience stores have the highest number of customers

In [ ]:
groupby_sector['인당이용금액'] = groupby_sector['이용금액'] / groupby_sector['이용자수'] # Calculate average spending per customer (sales amount / number of customers)

In [ ]:
groupby_sector.sort_values(by='이용자수', ascending=False).head(10)

In [ ]:
#Convenience stores have high customer count but low spending per customer

In [ ]:
groupby_sector.sort_values(by='인당이용금액', ascending=False).head(10) # Extract average spending per customer

In [ ]:
groupby_sector.sort_values(by='이용자수').head(10)

In [ ]:
# Having only 5 customers with the highest spending per customer doesn't make sense
jeju_reg_df = jeju_reg_df[jeju_reg_df['업종명'] != '버스 운송업']

In [ ]:
# Calculate total sales and average spending per customer by industry
groupby_sector = jeju_reg_df.groupby("업종명").sum(numeric_only = True)
groupby_sector['인당이용금액'] = groupby_sector['이용금액'] / groupby_sector['이용자수']

# Top 10 industries by average spending per customer
groupby_sector.sort_values(by='인당이용금액', ascending=False).head(10)

In [ ]:
# Bars/pubs have the highest average spending per customer

In [ ]:
groupby_reg = jeju_reg_df.groupby('읍면동명').sum(numeric_only=True) # group by area

In [ ]:
groupby_reg.sort_values(by='이용금액', ascending=False).head(10)

In [ ]:
# "연동" areas has the highest total sales amount

In [ ]:
groupby_reg.sort_values(by='이용자수', ascending=False).head(10)

In [ ]:
# "노형동" areas has the highest total customers

In [ ]:
groupby_reg['인당이용금액'] = groupby_reg['이용금액'] / groupby_reg['이용자수']
groupby_reg.sort_values(by='인당이용금액', ascending=False).iloc[:5] #Calculate average spending per customer by region and get top 5

In [ ]:
top5_region = groupby_reg.sort_values(by='인당이용금액', ascending=False).iloc[:5].index
top5_region

In [ ]:
groupby_reg_sec = jeju_reg_df.groupby(['읍면동명', '업종명']).sum(numeric_only=True).reset_index()

for reg in top5_region:
    reg_df = groupby_reg_sec[groupby_reg_sec['읍면동명'] == reg]
    print(reg, reg_df.sort_values(by='이용금액', ascending=False).iloc[:5]['업종명'].tolist()) # Check top 5 industries by sales amount for each of the top 5 regions

In [ ]:
# ----------------------------Analysis of top regions and industries by card usage complete----------------------

In [ ]:
jeju_pop_df.head()

In [ ]:
print_unique_values(jeju_pop_df)

In [ ]:
jeju_pop_df.groupby('읍면동명').sum(numeric_only=True).sort_values(by='방문인구', ascending=False).iloc[:10] # Top 10 areas with the highest visitor population

In [ ]:
jeju_reg_df.head()

In [ ]:
jeju_pop_df.head()

In [ ]:
#jeju_pop_df['연월일'].unique()

In [ ]:
jeju_pop_df['연월일'] = jeju_pop_df['연월일'].astype('string')
jeju_pop_df['연월'] = jeju_pop_df['연월일'].str[:4] + '-' + jeju_pop_df['연월일'].str[4:6] # Align column formats before joining datasets

In [ ]:
jeju_pop_df.head()

In [ ]:
jeju_pop_df['성별'] = jeju_pop_df['성별'] + '성'

In [ ]:
jeju_pop_df.head()

In [ ]:
groupby_pop = jeju_pop_df.groupby(['연월', '시군구명', '읍면동명', '성별']).sum(numeric_only=True).reset_index()
groupby_pop.head()

In [ ]:
jeju_df = pd.merge(jeju_reg_df, groupby_pop, how='left', on=['연월', '시군구명', '읍면동명', '성별'])
jeju_df.head() #Combine two datasets into one

In [ ]:
jeju_df['업종명'].unique()

In [ ]:
# 비알콜 음료점업 means cafe

In [ ]:
cafe_df = jeju_df[jeju_df['업종명'] == '비알콜 음료점업']
cafe_df.groupby('읍면동명').sum(numeric_only=True).sort_values(by='이용금액', ascending=False).iloc[:10] # Top 10 regions with the highest cafe sales amount

In [ ]:
sns.scatterplot(cafe_df, x='방문인구', y='이용금액')
plt.title('Visitor population and card sales amount for cafes')

In [ ]:
# Higher visitor population correlates with higher card sales amount

In [ ]:
cafe_df.corr(numeric_only=True) # check correlation

In [ ]:
sns.scatterplot(jeju_df, x='방문인구', y='이용금액')
plt.title('Visitor population and card sales amount across all industries')

In [ ]:
# No clear correlation observed

In [ ]:
jeju_df.corr(numeric_only=True)

In [ ]:
# ---------------------------Analysis of cafes and visitor population complete------------------

## Key Findings & Business Insights

### 1. Top Industries & Regions
- Korean restaurants ranked 1st in total sales, convenience stores had the most customers
- Bars/pubs showed the highest spending per customer
- Yeondong district had the highest total sales amount

### 2. Cafe & Visitor Population
- Cafe sales and visitor population showed a clear positive correlation (r ≈ 0.633)
- 6 out of top 10 cafe regions overlapped with top 10 high-traffic regions
- No clear correlation was observed across all industries (r ≈ 0.163)

## Conclusion
Based on card spending and visitor population data from 2017–2018,
cafes tend to perform better in high-traffic areas.
If opening a cafe in Jeju, prioritizing regions with high visitor population
such as Yeondong and Ido2-dong would be a data-backed strategy.